# 07 — Busca de hiperparâmetros do Random Forest binário (grade completa, validação cruzada de 5 dobras e sementes)

Este notebook responde: **quais hiperparâmetros do RF binário (sem os atributos de AUC alto) funcionam melhor, e quanto cada um influencia o resultado?** Depois treina o modelo final com a combinação escolhida, com o mesmo protocolo e o mesmo cálculo de custo do notebook 05. A avaliação no teste fica para o notebook 08.

| O que | Como |
|---|---|
| **Grade completa** | `n_estimators` × `max_depth` × `min_samples_leaf` (3 × 3 × 3 = **27 combinações**), todas cruzadas. Valores em `GRADE` |
| **Atalho das árvores** | Com a mesma semente, as primeiras *k* árvores de uma floresta maior são **idênticas** às de uma floresta de *k* árvores. Treina-se uma floresta com o maior `n_estimators` e avaliam-se as primeiras 100, 300 e 500. O notebook confere isso na sua versão do scikit-learn |
| **Validação cruzada** | 5 dobras no **ajuste** (balanceado 1:1), estratificadas por tipo de ataque. As dobras são **as mesmas** para todas as combinações (comparação pareada) |
| **Sementes** | 3 sementes do RF por combinação, para medir o ruído do sorteio do modelo |
| **Validação separada** | Cada combinação também é treinada no ajuste inteiro e avaliada na **validação** (60.154 linhas), com as métricas na proporção real |
| **Métricas** | As mesmas do notebook 05 (ROC-AUC, recall, falso alarme, MCC, precisão e F1 do BENIGN, PR-AUC etc., sem pesos e na proporção real) **mais o recall por tipo de ataque** |
| **Custo** | Para cada combinação: nós, profundidade, tamanho do modelo, tempo de treino e de previsão. Para o modelo final: o mesmo custo completo do notebook 05 (tempo, CPU, RAM, latência, vazão) |

## Regra de escolha (definida **antes** de ver os resultados)
1. Só a **validação cruzada** decide (o teste não entra, e só é usado uma vez, no notebook 08).
2. Duas métricas de decisão: **ROC-AUC** e **MCC na proporção real**, médias das dobras (e das sementes).
3. **Regra de um erro-padrão:** para cada métrica, toda combinação a **menos de 1 erro-padrão** da melhor conta como empatada com ela. Fica elegível quem empata **nas duas** métricas.
4. Entre as elegíveis, vence a **mais barata** (menos nós no total, que reflete tempo e memória).
5. Se nenhuma for elegível nas duas, vence a de melhor posição média nas duas métricas.

## O que o notebook grava
Uma pasta `artefatos_modelos/rf_binario_busca/rf_busca_<data>` com: os resultados de toda a grade (CSV), os efeitos de cada hiperparâmetro (CSV e gráficos), a combinação escolhida e por quê, o modelo do ajuste e o **modelo final (refit)**, as métricas e os limiares da validação, o recall por tipo e o custo computacional.

## Antes de rodar
- Os notebooks 02, 03, 04 e **05** precisam ter terminado (este usa as mesmas views e o mesmo refit).
- Feche os outros notebooks (cada kernel aberto guarda memória) e deixe o computador na tomada, sem suspensão.
- A busca é **longa** (a barra de progresso mostra o andamento e o tempo restante). Cada unidade pronta é salva: se algo interromper, **rode a célula de novo** e ela retoma de onde parou.
- Requisitos: `scikit-learn`, `psutil` e `matplotlib` (opcional, para os gráficos).

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


## 1. Parâmetros e o que existe
`GRADE` guarda os valores testados. `max_depth`: `None` (sem limite; no modelo do notebook 05 as árvores chegam a ~27 níveis, média ~21), 15 (corta as árvores mais fundas) e 8 (limite forte). O valor 30 foi dispensado porque nunca seria atingido e repetiria o `None`. `min_samples_leaf`: 1, 5 e 10. Para mudar algo, defina a variável antes desta célula (por exemplo, `GRADE = {...}`).

In [6]:
# ---------- 1.1 Parâmetros ----------
GRADE = globals().get("GRADE", {"n_estimators": (100, 300, 500), "max_depth": (None, 15, 8), "min_samples_leaf": (1, 5, 10)})
SEMENTES = tuple(globals().get("SEMENTES", (42, 43, 44)))             # sementes do RF (a aleatoriedade do modelo)
SEMENTE = globals().get("SEMENTE", 42)                                # dobras da validação cruzada, partições da validação, refit e amostras de latência
K_CV = globals().get("K_CV", 5)
K_PARTICOES_VAL = globals().get("K_PARTICOES_VAL", 5)
REPETICOES_TEMPO = globals().get("REPETICOES_TEMPO", 3)
N_LATENCIA = globals().get("N_LATENCIA", 1000)
FPR_ALVOS = globals().get("FPR_ALVOS", (0.01, 0.001))
LIMIAR = globals().get("LIMIAR", 0.5)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3)
TAMANHOS_CURVA = tuple(globals().get("TAMANHOS_CURVA", (25, 50, 100, 200, 300, 400, 500)))   # tamanhos extras (gratuitos) para a curva "desempenho × número de árvores"
MULT_ERRO_PADRAO = globals().get("MULT_ERRO_PADRAO", 1.0)
METRICAS_DECISAO = ("roc_auc", "mcc_real")
PADRAO = {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1}      # a configuração do notebook 05
PARAR_APOS_UNIDADES = globals().get("PARAR_APOS_UNIDADES", None)             # só para testes
NOVA_EXECUCAO = globals().get("NOVA_EXECUCAO", False)                        # True = não retomar uma busca interrompida
FIXOS = dict(min_samples_split=2, max_features="sqrt", n_jobs=-1)
assert set(GRADE) == {"n_estimators", "max_depth", "min_samples_leaf"} and all(len(v) >= 2 for v in GRADE.values()), "GRADE precisa ter os 3 hiperparâmetros, cada um com 2 valores ou mais"

TAM_GRADE = sorted(GRADE["n_estimators"]); NMAX = max(TAM_GRADE)
TAM_CURVA_USADOS = tuple(k for k in TAMANHOS_CURVA if k <= NMAX); TAM_TODOS = sorted(set(TAM_GRADE) | set(TAM_CURVA_USADOS))
rot = lambda d: "None" if d is None else str(d)
nome_unidade = lambda d, l, s: f"prof{rot(d)}_folha{l}_semente{s}"
PARES = [(d, l) for d in GRADE["max_depth"] for l in GRADE["min_samples_leaf"]]
UNIDADES = [(d, l, s) for (d, l) in PARES for s in SEMENTES]
CONFIGS = [(n, d, l) for n in TAM_GRADE for (d, l) in PARES]
N_FLORESTAS = len(UNIDADES) * (K_CV + 1)
ARTEFATOS_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_binario_busca"; ARTEFATOS_BASE.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT = "prep_vw_bal_bin_treino", "prep_vw_bal_bin_validacao", "prep_vw_bal_bin_refit"
SEL_TAB, DADOS_TAB, TAB_REFIT = "prep_bal_bin_selecao", "prep_bal_bin_dados", "prep_bal_bin_refit_sel"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, V_REFIT, SEL_TAB, DADOS_TAB, "prep_views_rf_binario") if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03, 04 e 05: faltam {faltam}"

versao_sel = str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])
aud = consultar_num("SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM prep_views_rf_binario ORDER BY posicao")
versao_views = str(aud["versao"].iloc[0])
N_BENIGN, N_ATAQUE = n_linhas("prep_final_treino", "label_binary = 0"), n_linhas("prep_final_treino", "label_binary = 1")
cont = consultar_num(f"SELECT papel, y, count() AS n FROM {SEL_TAB} GROUP BY papel, y"); cont = {(r.papel, int(r.y)): int(r.n) for r in cont.itertuples()}
n_aj_b, n_aj_a, n_va_b, n_va_a = cont[("treino", 0)], cont[("treino", 1)], cont[("validacao", 0)], cont[("validacao", 1)]
PESO_BENIGN_PROP = (N_BENIGN / N_ATAQUE) * (n_va_a / n_va_b)          # peso de cada BENIGN da validação para ela ter a proporção do treino
PESO_BENIGN_CV = (N_BENIGN / N_ATAQUE) * (n_aj_a / n_aj_b)            # idem para as linhas do ajuste (as dobras são 1:1)
print(f"Seleção do notebook 04 (versão {versao_sel}) | views do notebook 03 (versão {versao_views})")
print(f"Treino: {N_BENIGN:,} BENIGN e {N_ATAQUE:,} ataques | ajuste: {n_aj_b:,} + {n_aj_a:,} | validação: {n_va_b:,} + {n_va_a:,}")
print(f"Peso de cada BENIGN na proporção real: validação {PESO_BENIGN_PROP:.5f} | ajuste {PESO_BENIGN_CV:.5f} (ataques = 1)")
print(f"\nGrade: {len(CONFIGS)} configurações = {len(TAM_GRADE)} (n_estimators {TAM_GRADE}) × {len(GRADE['max_depth'])} (max_depth {[rot(d) for d in GRADE['max_depth']]}) × {len(GRADE['min_samples_leaf'])} (min_samples_leaf {list(GRADE['min_samples_leaf'])})")
print(f"Sementes: {SEMENTES} | validação cruzada: {K_CV} dobras")
print(f"Treinos: {len(PARES)} pares (profundidade × folha) × {len(SEMENTES)} sementes × ({K_CV} dobras + 1 treino completo) = {N_FLORESTAS} florestas de {NMAX} árvores")
print(f"(as florestas de {', '.join(str(k) for k in TAM_GRADE if k < NMAX)} árvores saem das primeiras árvores de cada floresta de {NMAX})")

Seleção do notebook 04 (versão 20261018_204854) | views do notebook 03 (versão 20261018_180729)
Treino: 50,768 BENIGN e 8,206,575 ataques | ajuste: 40,614 + 40,614 | validação: 10,154 + 50,000
Peso de cada BENIGN na proporção real: validação 0.03046 | ajuste 0.00619 (ataques = 1)

Grade: 27 configurações = 3 (n_estimators [100, 300, 500]) × 3 (max_depth ['None', '15', '8']) × 3 (min_samples_leaf [1, 5, 10])
Sementes: (42, 43, 44) | validação cruzada: 5 dobras
Treinos: 9 pares (profundidade × folha) × 3 sementes × (5 dobras + 1 treino completo) = 162 florestas de 500 árvores
(as florestas de 100, 300 árvores saem das primeiras árvores de cada floresta de 500)


## 2. Dados, pré-processamento e dobras
Lê o ajuste e a validação (as mesmas views do notebook 05), **ordena as linhas por `id_linha`** (assim as dobras e os treinos são reproduzíveis e a busca pode ser retomada) e junta o **tipo de ataque** de cada linha (da tabela de seleção do notebook 04). As dobras da validação cruzada são estratificadas por tipo (BENIGN + cada tipo de ataque), então todo tipo aparece em todas as dobras. A célula também define a pasta da execução: se existir uma busca **interrompida com a mesma configuração**, ela é retomada.

In [7]:
# ---------- 2.1 Dados, pré-processamento, tipos e dobras ----------
import hashlib
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views)
assert prep.colunas_modelo == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
print(f"Atributos do modelo: {len(prep.colunas_modelo)} ({len(prep.features)} + {len(prep.indicadores)} indicadores) | retiradas na ablação: {RETIRADAS}")

def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID))
CM = prep.colunas_modelo
X_aj, y_aj = aj[CM].to_numpy(float), aj["y"].to_numpy(int)
X_va, y_va = va[CM].to_numpy(float), va["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
X_va_raw = X_va[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_va_raw[X_va[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_va_raw), X_va, rtol=1e-12, atol=1e-12), "o PreprocessadorRF não reproduz a matriz da view"

# tipo de ataque de cada linha (da seleção do notebook 04); o prefixo DrDoS_ é retirado dos nomes
nome_tipo = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
sel = consultar(f"SELECT toString(id_linha) AS id_linha, papel, tipo FROM {SEL_TAB}"); sel["tipo"] = sel["tipo"].map(nome_tipo)
tipo_aj = aj[["id_linha"]].merge(sel[sel.papel == "treino"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]
tipo_va = va[["id_linha"]].merge(sel[sel.papel == "validacao"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]
assert tipo_aj.notna().all() and tipo_va.notna().all(), "há linhas sem tipo de ataque na tabela de seleção"
TIPOS = ["BENIGN"] + sorted(t for t in set(tipo_aj) | set(tipo_va) if t != "BENIGN"); COD = {t: i for i, t in enumerate(TIPOS)}
g_aj, g_va = tipo_aj.map(COD).to_numpy(np.int16), tipo_va.map(COD).to_numpy(np.int16)
assert ((g_aj == 0) == (y_aj == 0)).all() and ((g_va == 0) == (y_va == 0)).all(), "o tipo BENIGN não coincide com y = 0"

DOBRAS = list(StratifiedKFold(K_CV, shuffle=True, random_state=SEMENTE).split(X_aj, g_aj))
assert all(len(np.unique(g_aj[ite])) == len(TIPOS) for _, ite in DOBRAS), "alguma dobra ficou sem algum tipo de ataque"
tab_tipos = pd.DataFrame({"ajuste": np.bincount(g_aj, minlength=len(TIPOS)), "validação": np.bincount(g_va, minlength=len(TIPOS))}, index=TIPOS)
print(f"\nAjuste: {len(X_aj):,} linhas ({int((y_aj == 0).sum()):,} BENIGN) | validação: {len(X_va):,} ({int((y_va == 0).sum()):,} BENIGN) | {len(TIPOS) - 1} tipos de ataque")
display(tab_tipos.T)
print(f"Dobras: {K_CV}, estratificadas por tipo (cada dobra de avaliação tem ~{len(X_aj) // K_CV:,} linhas)")

# pasta da execução: retoma uma busca interrompida com a MESMA configuração e os MESMOS dados
crc = lambda s: zlib.crc32(",".join(s).encode())
IMPRESSAO = hashlib.sha1(json.dumps(_limpar({"grade": GRADE, "sementes": SEMENTES, "semente": SEMENTE, "k_cv": K_CV, "tam_curva": TAM_CURVA_USADOS, "fixos": FIXOS, "versao_sel": versao_sel,
                                           "versao_views": versao_views, "ids_aj": crc(aj["id_linha"]), "ids_va": crc(va["id_linha"])}), sort_keys=True).encode()).hexdigest()[:12]
PASTA = None
if not NOVA_EXECUCAO:
    for p in sorted(ARTEFATOS_BASE.glob("rf_busca_*"), reverse=True):
        try:
            if not (p / "FINALIZADO").exists() and json.loads((p / "config_busca.json").read_text(encoding="utf-8"))["impressao"] == IMPRESSAO: PASTA = p; break
        except (OSError, ValueError, KeyError): pass
RETOMANDO = PASTA is not None
if PASTA is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA = ARTEFATOS_BASE / f"rf_busca_{RUN_ID}"; PASTA.mkdir(parents=True)
    salvar_json({"run_id": RUN_ID, "criado_em": datetime.now().isoformat(timespec="seconds"), "impressao": IMPRESSAO, "grade": GRADE, "sementes": SEMENTES}, PASTA / "config_busca.json")
else:
    RUN_ID = PASTA.name.replace("rf_busca_", "")
(PASTA / "unidades").mkdir(exist_ok=True)
print(f"\nPasta da execução: {PASTA}" + ("  (RETOMANDO uma busca interrompida)" if RETOMANDO else ""))

Atributos do modelo: 43 (42 + 1 indicadores) | retiradas na ablação: ['bwd_packets_s', 'inbound', 'total_backward_packets']

Ajuste: 81,228 linhas (40,614 BENIGN) | validação: 60,154 (10,154 BENIGN) | 12 tipos de ataque


,BENIGN,DNS,LDAP,MSSQL,NTP,NetBIOS,SNMP,SSDP,Syn,TFTP,UDP,UDPLag,WebDDoS
ajuste,40614,3662,3662,3662,3662,3662,3662,3662,3662,3663,3662,3662,331
validação,10154,4538,4538,4538,4538,4537,4538,4538,4538,4538,4538,4538,83


Dobras: 5, estratificadas por tipo (cada dobra de avaliação tem ~16,245 linhas)

Pasta da execução: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_busca\rf_busca_20261024_175105


### Ferramentas da busca
- **`Barra`**: barra de progresso (percentual, contagem, tempo decorrido e tempo restante estimado). Funciona no Jupyter e no VS Code sem instalar nada.
- **`prever_prefixos`**: probabilidade de ataque usando as primeiras *k* árvores de uma floresta, para vários *k*, numa só passada.
- **`recall_por_tipo`**: recall de cada tipo de ataque (e, na primeira posição, a especificidade do BENIGN).

In [8]:
import copy, io, gc

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

def _sub_floresta(rf, ini, fim):
    sub = copy.copy(rf); sub.estimators_ = rf.estimators_[ini:fim]; sub.n_estimators = fim - ini; return sub

def prever_prefixos(rf, X, tamanhos):
    # Probabilidade de ataque usando as primeiras k árvores, para cada k de `tamanhos` (crescente), numa só passada pelas árvores.
    # Devolve ({k: probabilidades float32}, {k: tempo acumulado de previsão em s}).
    X32 = np.ascontiguousarray(X, dtype=np.float32); soma = np.zeros(len(X32)); saida, tempos, ant, acum = {}, {}, 0, 0.0
    for k in tamanhos:
        t0 = time.perf_counter(); p = _sub_floresta(rf, ant, k).predict_proba(X32)[:, 1]; acum += time.perf_counter() - t0
        soma += p * (k - ant); saida[k] = (soma / k).astype(np.float32); tempos[k] = acum; ant = k
    return saida, tempos

def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

# autoteste (não toca no servidor): barra, prefixos e recall por tipo
_rf = RandomForestClassifier(n_estimators=30, random_state=1, n_jobs=1).fit(np.random.default_rng(0).normal(size=(300, 4)), np.random.default_rng(1).integers(0, 2, 300))
_X = np.random.default_rng(2).normal(size=(50, 4)); _pf, _ = prever_prefixos(_rf, _X, [10, 30])
assert np.allclose(_pf[30], _rf.predict_proba(_X.astype(np.float32))[:, 1], atol=1e-6), "prever_prefixos não reproduz o predict_proba"
_r = recall_por_tipo(np.array([0, 0, 1, 1, 1]), np.array([.1, .9, .8, .2, .7]), np.array([0, 0, 1, 1, 2]), 0.5, 3); assert np.allclose(_r, [0.5, 0.5, 1.0]), _r
_b = Barra(2, "autoteste"); _b.avancar(2); assert fmt_tempo(3725) == "1h 02min"
print("Autoteste das ferramentas da busca OK")

Autoteste das ferramentas da busca OK


## 3. A busca (a parte longa)
Para cada par (`max_depth`, `min_samples_leaf`) e cada semente, uma **unidade** treina:
- **5 florestas** (uma por dobra), cada uma em 4/5 do ajuste e avaliada no 1/5 que ficou de fora (as previsões "fora da dobra" de todas as dobras cobrem o ajuste inteiro uma vez);
- **1 floresta** no ajuste inteiro, avaliada na validação, com o custo medido (nós, profundidade, tamanho do modelo, tempo de previsão, pico de RAM).

Cada floresta tem o maior `n_estimators` (500) e é avaliada com as primeiras 100, 300 e 500 árvores (e mais alguns tamanhos para a curva "desempenho × árvores"). **Cada unidade pronta é salva** em `unidades/`: se a busca for interrompida, rode a célula de novo e ela retoma. A barra mostra o andamento e o tempo restante estimado.

In [9]:
# ---------- 3.1 Busca ----------
# conferência rápida ANTES da parte longa: as primeiras k árvores de uma floresta maior = floresta treinada com k árvores (na sua versão do scikit-learn)
_i = np.random.default_rng(0).choice(len(X_aj), size=min(3000, len(X_aj)), replace=False)
_g = RandomForestClassifier(n_estimators=40, random_state=7, **FIXOS).fit(X_aj[_i], y_aj[_i]); _p, _ = prever_prefixos(_g, X_va[:2000], [20, 40])
_d = RandomForestClassifier(n_estimators=20, random_state=7, **FIXOS).fit(X_aj[_i], y_aj[_i])
assert np.allclose(_p[20], _d.predict_proba(X_va[:2000])[:, 1], atol=1e-6), "o atalho das árvores NÃO vale nesta versão do scikit-learn: não rode a busca e avise antes"
print(f"✔ Atalho das árvores conferido (scikit-learn {sklearn.__version__}): as primeiras 20 de 40 árvores = floresta treinada com 20")
del _g, _d, _p
prontas = sum((PASTA / "unidades" / f"{nome_unidade(*u)}.npz").exists() for u in UNIDADES)
print(f"Unidades prontas: {prontas}/{len(UNIDADES)}" + ("  (retomando)" if prontas else "") + f" | florestas a treinar agora: {(len(UNIDADES) - prontas) * (K_CV + 1)}")
barra = Barra(N_FLORESTAS, "Busca de hiperparâmetros"); barra.n = prontas * (K_CV + 1); barra._mostrar(forcar=True); novas = 0
t_sessao, inicio_sessao, interrompida = time.perf_counter(), datetime.now().isoformat(timespec="seconds"), True
try:
    for (d, l, s) in UNIDADES:
        arq = PASTA / "unidades" / f"{nome_unidade(d, l, s)}.npz"
        if arq.exists(): continue
        hp = dict(n_estimators=NMAX, max_depth=d, min_samples_leaf=l, random_state=s, **FIXOS); rotulo = f"prof={rot(d)} folha={l} semente={s}"
        oof = np.full((len(TAM_GRADE), len(y_aj)), np.nan, np.float32); curva_cv = np.zeros((len(DOBRAS), len(TAM_CURVA_USADOS))); t_cv = []
        for f, (itr, ite) in enumerate(DOBRAS):
            t0 = time.perf_counter(); m_ = RandomForestClassifier(**hp).fit(X_aj[itr], y_aj[itr]); t_cv.append(time.perf_counter() - t0)
            pf, _ = prever_prefixos(m_, X_aj[ite], TAM_TODOS)
            for i, k in enumerate(TAM_GRADE): oof[i, ite] = pf[k]
            curva_cv[f] = [roc_auc_score(y_aj[ite], pf[k]) for k in TAM_CURVA_USADOS]
            del m_, pf; barra.avancar(1, f"{rotulo} | dobra {f + 1}/{len(DOBRAS)} ({t_cv[-1]:.0f} s)")
        with Medidor() as med: m_ = RandomForestClassifier(**hp).fit(X_aj, y_aj)
        pv, tv = prever_prefixos(m_, X_va, TAM_TODOS); por_tam = {}
        for k in TAM_GRADE:
            sub = _sub_floresta(m_, 0, k); buf = io.BytesIO(); joblib.dump(sub, buf, compress=COMPRESSAO_MODELO)
            por_tam[str(k)] = {"nos": int(sum(e.tree_.node_count for e in sub.estimators_)), "prof_media": float(np.mean([e.tree_.max_depth for e in sub.estimators_])),
                               "prof_max": int(max(e.tree_.max_depth for e in sub.estimators_)), "t_prev_val_s": float(tv[k]), "modelo_mib": buf.tell() / 2 ** 20}
        meta = {"max_depth": d, "min_samples_leaf": l, "semente": s, "t_fit_dobras_s": t_cv, "t_fit_completo_s": med.relogio, "cpu_fit_completo_s": med.cpu, "pico_ram_mib": med.pico_mib,
                "acrescimo_ram_mib": med.delta_mib, "por_tamanho": por_tam, "tam_curva": list(TAM_CURVA_USADOS), "tam_grade": TAM_GRADE}
        curva_val = np.array([roc_auc_score(y_va, pv[k]) for k in TAM_CURVA_USADOS])
        tmp = arq.with_name(arq.stem + ".tmp.npz")
        np.savez_compressed(tmp, oof=oof, val=np.stack([pv[k] for k in TAM_GRADE]), curva_cv=curva_cv, curva_val=curva_val, meta=json.dumps(_limpar(meta))); os.replace(tmp, arq)
        del m_, sub, pv; gc.collect(); barra.avancar(1, f"{rotulo} | treino completo ({med.relogio:.0f} s) | unidade salva"); novas += 1
        if PARAR_APOS_UNIDADES and novas >= PARAR_APOS_UNIDADES: raise RuntimeError("parada de teste (PARAR_APOS_UNIDADES)")
except KeyboardInterrupt:
    print(f"\nInterrompido. As unidades prontas ficam salvas ({sum((PASTA / 'unidades' / f'{nome_unidade(*u)}.npz').exists() for u in UNIDADES)}/{len(UNIDADES)}): rode esta célula de novo para retomar.")
    raise
else:
    interrompida = False
finally:                                                         # grava o tempo desta sessão mesmo se a busca for interrompida ou falhar
    rec = {"inicio": inicio_sessao, "fim": datetime.now().isoformat(timespec="seconds"), "relogio_s": time.perf_counter() - t_sessao, "unidades_novas": novas, "florestas_novas": novas * (K_CV + 1), "interrompida": interrompida}
    pd.DataFrame([rec]).to_csv(PASTA / "tempos_busca.csv", mode="a", header=not (PASTA / "tempos_busca.csv").exists(), index=False)
barra.fechar()
print(f"✔ Busca concluída: {len(UNIDADES)} unidades em {PASTA / 'unidades'} | tempo desta sessão: {fmt_tempo(rec['relogio_s'])}")

✔ Atalho das árvores conferido (scikit-learn 1.9.1): as primeiras 20 de 40 árvores = floresta treinada com 20
Unidades prontas: 0/27 | florestas a treinar agora: 162


Busca de hiperparâmetros: concluído em 1h 08min
✔ Busca concluída: 27 unidades em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_busca\rf_busca_20261024_175105\unidades | tempo desta sessão: 1h 08min


## 4. Métricas de cada configuração
Lê as previsões gravadas e calcula, para cada configuração (27) e cada semente:
- **Validação cruzada**, dobra a dobra (média das sementes em cada dobra → média e **erro-padrão** entre as 5 dobras). As métricas "na proporção real" usam o peso do BENIGN do ajuste (0,00619), para a proporção BENIGN:ataque ser a do treino.
- **Previsões fora da dobra reunidas** (as 81.228 linhas do ajuste, cada uma prevista pelo modelo que não a viu), com recall e falso alarme nos pontos de operação de 1% e 0,1%.
- **Validação separada** (60.154 linhas), sem pesos e na proporção real.
- **Recall por tipo de ataque** (validação cruzada e validação).
- **Custo** de cada configuração (nós, profundidade, tamanho do modelo, tempo de previsão).
O **ruído** é medido de dois jeitos: o erro-padrão entre dobras e o desvio entre sementes.

In [10]:
# ---------- 4.1 Métricas por configuração ----------
faltam_u = [nome_unidade(*u) for u in UNIDADES if not (PASTA / "unidades" / f"{nome_unidade(*u)}.npz").exists()]
assert not faltam_u, f"faltam {len(faltam_u)} unidades: rode a célula 3.1 de novo (ela retoma de onde parou)"
M_BASE = ("roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "mcc")
M_REAL = ("mcc", "precisao_benign", "f1_benign", "pr_auc_benign", "f1_macro", "precisao", "f1")
linhas_fold, linhas_oof, linhas_val, linhas_tipo, linhas_curva, meta_u = [], [], [], [], [], {}
w_oof = np.where(y_aj == 0, PESO_BENIGN_CV, 1.0); w_val = np.where(y_va == 0, PESO_BENIGN_PROP, 1.0)
barra = Barra(len(UNIDADES), "Calculando as métricas")
for (d, l, s) in UNIDADES:
    z = np.load(PASTA / "unidades" / f"{nome_unidade(d, l, s)}.npz"); oof, val, meta = z["oof"], z["val"], json.loads(str(z["meta"])); meta_u[(d, l, s)] = meta
    assert not np.isnan(oof).any(), f"previsões fora da dobra incompletas em {nome_unidade(d, l, s)}"
    for i, k in enumerate(TAM_GRADE):
        base = {"n_estimators": k, "profundidade": rot(d), "min_samples_leaf": l, "semente": s}
        for f, (_, ite) in enumerate(DOBRAS, 1):
            yf, pf = y_aj[ite], oof[i, ite]; wf = np.where(yf == 0, PESO_BENIGN_CV, 1.0); a, b = metricas(yf, pf, LIMIAR, None), metricas(yf, pf, LIMIAR, wf)
            linhas_fold.append({**base, "dobra": f, **{m: a[m] for m in M_BASE}, **{m + "_real": b[m] for m in M_REAL}})
        a, b = metricas(y_aj, oof[i], LIMIAR, None), metricas(y_aj, oof[i], LIMIAR, w_oof)
        linhas_oof.append({**base, **{m: a[m] for m in M_BASE}, **{m + "_real": b[m] for m in M_REAL}, **{f"recall_fpr{al * 100:g}pct": recall_em_fpr(y_aj, oof[i], al)["recall"] for al in FPR_ALVOS}})
        a, b = metricas(y_va, val[i], LIMIAR, None), metricas(y_va, val[i], LIMIAR, w_val)
        linhas_val.append({**base, **{m: a[m] for m in M_BASE}, **{m + "_real": b[m] for m in M_REAL}, **{f"recall_fpr{al * 100:g}pct": recall_em_fpr(y_va, val[i], al)["recall"] for al in FPR_ALVOS}})
        for fonte, yy, pp, gg in (("cv_fora_da_dobra", y_aj, oof[i], g_aj), ("validacao", y_va, val[i], g_va)):
            for t, r in zip(TIPOS, recall_por_tipo(yy, pp, gg, LIMIAR, len(TIPOS))): linhas_tipo.append({**base, "fonte": fonte, "tipo": t, "recall": r})
    for j, k in enumerate(meta["tam_curva"]):
        linhas_curva += [{"profundidade": rot(d), "min_samples_leaf": l, "semente": s, "n_arvores": k, "fonte": "cv", "roc_auc": float(z["curva_cv"][:, j].mean())},
                         {"profundidade": rot(d), "min_samples_leaf": l, "semente": s, "n_arvores": k, "fonte": "validacao", "roc_auc": float(z["curva_val"][j])}]
    barra.avancar(1)
barra.fechar()
TEMPO_TREINO_SOMADO = float(sum(sum(m["t_fit_dobras_s"]) + m["t_fit_completo_s"] for m in meta_u.values()))      # soma dos tempos de treino de todas as florestas
tempos_sessoes = pd.read_csv(PASTA / "tempos_busca.csv") if (PASTA / "tempos_busca.csv").exists() else pd.DataFrame(columns=["relogio_s"])
TEMPO_BUSCA_RELOGIO = float(tempos_sessoes["relogio_s"].sum()) if len(tempos_sessoes) else float("nan")
print(f"\n⏱ Tempo da busca: {fmt_tempo(TEMPO_BUSCA_RELOGIO)} de relógio em {len(tempos_sessoes)} sessão(ões) (treino, previsões e gravações) | soma dos tempos de treino das {N_FLORESTAS} florestas: {fmt_tempo(TEMPO_TREINO_SOMADO)}")
df_fold, df_oof, df_val, df_tipo, df_curva = (pd.DataFrame(x) for x in (linhas_fold, linhas_oof, linhas_val, linhas_tipo, linhas_curva))
CHAVE = ["n_estimators", "profundidade", "min_samples_leaf"]

def resumo_cv(metrica):
    # média das sementes em cada dobra -> média e erro-padrão entre as dobras; desvio entre sementes (da média das dobras)
    por_dobra = df_fold.groupby(CHAVE + ["dobra"])[metrica].mean().reset_index(); g = por_dobra.groupby(CHAVE)[metrica]
    por_sem = df_fold.groupby(CHAVE + ["semente"])[metrica].mean().reset_index().groupby(CHAVE)[metrica].std(ddof=1)
    return pd.DataFrame({f"cv_{metrica}": g.mean(), f"cv_{metrica}_ep": g.std(ddof=1) / np.sqrt(len(DOBRAS)), f"cv_{metrica}_dp_semente": por_sem})
M_CV = ["roc_auc", "recall", "taxa_falso_alarme", "mcc", "mcc_real", "precisao_benign_real", "f1_benign_real", "pr_auc_benign_real"]
df_cfg = pd.concat([resumo_cv(m) for m in M_CV], axis=1)
M_VAL = ["roc_auc", "recall", "taxa_falso_alarme", "mcc_real", "precisao_benign_real", "f1_benign_real", "pr_auc_benign_real", "recall_fpr1pct", "recall_fpr0.1pct"]
gv = df_val.groupby(CHAVE)[M_VAL]; df_cfg = df_cfg.join(gv.mean().add_prefix("val_")).join(gv.std(ddof=1).add_prefix("val_dp_"))
go = df_oof.groupby(CHAVE)[["mcc_real", "recall_fpr1pct", "recall_fpr0.1pct"]].mean().add_prefix("oof_"); df_cfg = df_cfg.join(go)
custo_cfg = []
for (n, d, l) in CONFIGS:
    ms = [meta_u[(d, l, s)] for s in SEMENTES]; pt = [m["por_tamanho"][str(n)] for m in ms]
    custo_cfg.append({"n_estimators": n, "profundidade": rot(d), "min_samples_leaf": l, "nos": float(np.mean([p["nos"] for p in pt])), "prof_media": float(np.mean([p["prof_media"] for p in pt])),
                      "prof_max": int(max(p["prof_max"] for p in pt)), "modelo_mib": float(np.mean([p["modelo_mib"] for p in pt])), "t_prev_val_s": float(np.mean([p["t_prev_val_s"] for p in pt])),
                      "t_treino_s": float(np.mean([m["t_fit_completo_s"] for m in ms])) * n / NMAX, "pico_ram_mib": float(np.max([m["pico_ram_mib"] for m in ms]))})
df_cfg = df_cfg.reset_index().merge(pd.DataFrame(custo_cfg), on=CHAVE); df_cfg.insert(0, "config", [f"{n}|{p}|{l}" for n, p, l in zip(df_cfg.n_estimators, df_cfg.profundidade, df_cfg.min_samples_leaf)])
print(f"✔ {len(df_cfg)} configurações | {len(df_fold):,} medidas por dobra | {len(df_val)} medidas na validação")
cols_ver = ["n_estimators", "profundidade", "min_samples_leaf", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc_real", "cv_mcc_real_ep", "cv_recall", "cv_taxa_falso_alarme", "val_roc_auc", "val_mcc_real", "nos", "t_treino_s"]
print("\n== Todas as configurações (CV: média das dobras e sementes; ep = erro-padrão entre dobras; val = validação separada, média das sementes) ==")
display(df_cfg[cols_ver].round(5).sort_values(["n_estimators", "profundidade", "min_samples_leaf"]).reset_index(drop=True))

Calculando as métricas: concluído em 34s

⏱ Tempo da busca: 1h 08min de relógio em 1 sessão(ões) (treino, previsões e gravações) | soma dos tempos de treino das 162 florestas: 1h 06min
✔ 27 configurações | 405 medidas por dobra | 81 medidas na validação

== Todas as configurações (CV: média das dobras e sementes; ep = erro-padrão entre dobras; val = validação separada, média das sementes) ==


,n_estimators,profundidade,min_samples_leaf,cv_roc_auc,cv_roc_auc_ep,cv_mcc_real,cv_mcc_real_ep,cv_recall,cv_taxa_falso_alarme,val_roc_auc,val_mcc_real,nos,t_treino_s
0,100,15,1,0.99998,0.00001,0.92382,0.02258,0.99890,0.00106,1.00000,0.94022,38247.33333,5.04682
1,100,15,5,0.99998,0.00001,0.88493,0.02041,0.99826,0.00118,0.99999,0.92179,27880.00000,4.92563
2,100,15,10,0.99998,0.00001,0.86458,0.01949,0.99789,0.00144,0.99999,0.90114,23448.66667,4.93463
3,100,8,1,0.99996,0.00001,0.80152,0.00589,0.99661,0.00222,0.99998,0.82598,14733.33333,4.29519
4,100,8,5,0.99996,0.00001,0.78796,0.00848,0.99627,0.00227,0.99998,0.81136,13858.00000,4.30952
5,100,8,10,0.99995,0.00001,0.78538,0.00851,0.99621,0.00231,0.99998,0.80573,13173.33333,4.34847
6,100,None,1,0.99996,0.00004,0.93007,0.02243,0.99900,0.00106,0.99996,0.94021,42328.00000,5.29252
7,100,None,5,0.99998,0.00001,0.88674,0.01852,0.99830,0.00117,0.99999,0.92139,28542.66667,5.03865
8,100,None,10,0.99998,0.00001,0.86545,0.01945,0.99791,0.00144,0.99999,0.90038,23787.33333,4.80780
9,300,15,1,0.99999,0.00000,0.92818,0.02123,0.99897,0.00103,0.99999,0.94018,113642.66667,15.14046


## 5. Escolha da combinação (regra fixada antes)
Aplica a regra da introdução, **só com a validação cruzada**:
1. Para cada métrica de decisão (ROC-AUC e MCC na proporção real), acha a melhor configuração e o seu **erro-padrão** entre as 5 dobras.
2. Uma configuração **empata** com a melhor numa métrica se ficar a menos de 1 erro-padrão dela. Fica **elegível** quem empata nas duas.
3. Entre as elegíveis, vence a **mais barata** (menos nós no total). Desempates: menos árvores, depois folha mínima maior.
4. Se nenhuma for elegível nas duas, vence a de melhor posição média nas duas métricas (a mensagem avisa).

A **validação separada** (que não participa da decisão) é mostrada ao lado, só como conferência.

In [11]:
# ---------- 5.1 Escolha ----------
def escolher(df, metricas_dec=METRICAS_DECISAO, mult=MULT_ERRO_PADRAO):
    d = df.reset_index(drop=True).copy(); elegivel = np.ones(len(d), bool); info = {}
    for m in metricas_dec:
        i = int(d[f"cv_{m}"].idxmax()); lim = d.loc[i, f"cv_{m}"] - mult * d.loc[i, f"cv_{m}_ep"]
        d[f"empata_{m}"] = d[f"cv_{m}"] >= lim; elegivel &= d[f"empata_{m}"].to_numpy()
        info[m] = {"melhor": d.loc[i, "config"], "valor_melhor": float(d.loc[i, f"cv_{m}"]), "erro_padrao_melhor": float(d.loc[i, f"cv_{m}_ep"]), "limite": float(lim), "n_empatadas": int(d[f"empata_{m}"].sum())}
    fallback = not elegivel.any()
    if fallback:
        rk = sum(d[f"cv_{m}"].rank(ascending=False) for m in metricas_dec); elegivel = (rk == rk.min()).to_numpy()
    cand = d[elegivel].sort_values(["nos", "n_estimators", "min_samples_leaf"], ascending=[True, True, False]).reset_index(drop=True)
    return cand.iloc[0], cand, info, fallback

ESC, CAND, INFO_SEL, FALLBACK = escolher(df_cfg)
PROF_REAL = {rot(d): d for d in GRADE["max_depth"]}
N_ESC, D_ESC, L_ESC = int(ESC["n_estimators"]), PROF_REAL[ESC["profundidade"]], int(ESC["min_samples_leaf"])
HIPERPARAMETROS_FINAL = dict(n_estimators=N_ESC, max_depth=D_ESC, min_samples_split=2, min_samples_leaf=L_ESC, max_features="sqrt", random_state=SEMENTE, n_jobs=-1)

print("== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==")
display(pd.DataFrame(INFO_SEL).T.rename(columns={"melhor": "melhor configuração (n|prof|folha)", "valor_melhor": "valor", "erro_padrao_melhor": "erro-padrão", "limite": "limite de empate", "n_empatadas": "configs que empatam"}))
print(f"\n== Passo 2: elegíveis (empatam nas duas métricas): {len(CAND)} de {len(df_cfg)} ==" + ("   ⚠ NENHUMA empatou nas duas: usei a melhor posição média (plano B)" if FALLBACK else ""))
cols = ["config", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc_real", "cv_mcc_real_ep", "val_roc_auc", "val_mcc_real", "nos", "modelo_mib", "t_treino_s"]
display(CAND[cols].round(5).head(15))
print(f"\n✔ ESCOLHIDA: n_estimators={N_ESC}, max_depth={D_ESC}, min_samples_leaf={L_ESC}  (a mais barata entre as {len(CAND)} elegíveis: {ESC['nos']:,.0f} nós)")

def _linha(rotulo, r): return {"configuração": rotulo, "config (n|prof|folha)": r["config"], **{c: r[c] for c in ("cv_roc_auc", "cv_roc_auc_ep", "cv_mcc_real", "cv_mcc_real_ep", "cv_recall", "cv_taxa_falso_alarme", "val_roc_auc", "val_mcc_real", "val_recall_fpr1pct", "nos", "modelo_mib", "t_treino_s", "t_prev_val_s")}}
comp = [_linha("escolhida", ESC)]
pd_ = df_cfg[(df_cfg.n_estimators == PADRAO["n_estimators"]) & (df_cfg.profundidade == rot(PADRAO["max_depth"])) & (df_cfg.min_samples_leaf == PADRAO["min_samples_leaf"])]
if len(pd_): comp.append(_linha("padrão do notebook 05", pd_.iloc[0]))
for m in METRICAS_DECISAO: comp.append(_linha(f"melhor em {m} (CV)", df_cfg.loc[df_cfg[f"cv_{m}"].idxmax()]))
print("\n== A escolhida × o padrão × as melhores de cada métrica ==")
tab_comp = pd.DataFrame(comp); display(tab_comp.round(5))
rk_val = {m: int((df_cfg[f"val_{m}"] > ESC[f"val_{m}"]).sum()) + 1 for m in ("roc_auc", "mcc_real")}
print(f"Conferência na validação separada (não decide nada): a escolhida ficou em {rk_val['roc_auc']}º de {len(df_cfg)} em ROC-AUC e em {rk_val['mcc_real']}º em MCC na proporção real.")

== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==


,melhor configuração (n|prof|folha),valor,erro-padrão,limite de empate,configs que empatam
roc_auc,300|15|1,0.999991,0.000004,0.999986,2
mcc_real,300|None|1,0.934044,0.021363,0.91268,6



== Passo 2: elegíveis (empatam nas duas métricas): 2 de 27 ==


,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc_real,cv_mcc_real_ep,val_roc_auc,val_mcc_real,nos,modelo_mib,t_treino_s
0,300|15|1,0.99999,0.0,0.92818,0.02123,0.99999,0.94018,113642.66667,2.23450,15.14046
1,500|15|1,0.99999,0.0,0.92973,0.02101,0.99999,0.94105,189531.33333,3.72441,25.23410



✔ ESCOLHIDA: n_estimators=300, max_depth=15, min_samples_leaf=1  (a mais barata entre as 2 elegíveis: 113,643 nós)

== A escolhida × o padrão × as melhores de cada métrica ==


,configuração,config (n|prof|folha),cv_roc_auc,cv_roc_auc_ep,cv_mcc_real,cv_mcc_real_ep,cv_recall,cv_taxa_falso_alarme,val_roc_auc,val_mcc_real,val_recall_fpr1pct,nos,modelo_mib,t_treino_s,t_prev_val_s
0,escolhida,300|15|1,0.99999,0.00000,0.92818,0.02123,0.99897,0.00103,0.99999,0.94018,0.99997,113642.66667,2.23450,15.14046,0.66995
1,padrão do notebook 05,100|None|1,0.99996,0.00004,0.93007,0.02243,0.99900,0.00106,0.99996,0.94021,0.99995,42328.00000,0.80871,5.29252,0.31282
2,melhor em roc_auc (CV),300|15|1,0.99999,0.00000,0.92818,0.02123,0.99897,0.00103,0.99999,0.94018,0.99997,113642.66667,2.23450,15.14046,0.66995
3,melhor em mcc_real (CV),300|None|1,0.99997,0.00003,0.93404,0.02136,0.99906,0.00101,0.99999,0.94023,0.99996,127299.33333,2.42990,15.87757,0.77525


Conferência na validação separada (não decide nada): a escolhida ficou em 4º de 27 em ROC-AUC e em 5º em MCC na proporção real.


## 6. A influência de cada hiperparâmetro
Como a grade é **completa** (todas as combinações), dá para medir cada hiperparâmetro separadamente e também o efeito de um sobre o outro:
- **Efeito principal:** a média da métrica em cada valor do hiperparâmetro, olhando todas as combinações dos outros dois. A **amplitude** (maior média − menor média) diz o quanto ele mexe no resultado, e é comparada com o **ruído** (erro-padrão entre dobras): só vale como efeito se for bem maior que o ruído.
- **Um de cada vez:** a diferença para o padrão (100 árvores, sem limite, folha 1) mudando **só** um hiperparâmetro.
- **Interação:** o mapa de calor de `max_depth` × `min_samples_leaf` mostra se o efeito de um depende do outro (os dois limitam a complexidade da árvore).
- **Decomposição da variação:** que fração da variação entre as 27 configurações vem de cada hiperparâmetro e de cada interação (soma 100%).
- **Curva de estabilização:** o ROC-AUC conforme o número de árvores.

In [12]:
# ---------- 6.1 Influência dos hiperparâmetros ----------
INFL = [("ROC-AUC (CV)", "cv_roc_auc"), ("MCC proporção real (CV)", "cv_mcc_real"), ("recall (CV)", "cv_recall"), ("falso alarme (CV)", "cv_taxa_falso_alarme"),
        ("ROC-AUC (validação)", "val_roc_auc"), ("MCC proporção real (validação)", "val_mcc_real"), ("recall com 1% de falso alarme (validação)", "val_recall_fpr1pct"),
        ("nós totais", "nos"), ("tamanho do modelo (MiB)", "modelo_mib"), ("tempo de treino (s)", "t_treino_s"), ("tempo de previsão na validação (s)", "t_prev_val_s")]
NIVEIS = {"n_estimators": TAM_GRADE, "profundidade": [rot(d) for d in GRADE["max_depth"]], "min_samples_leaf": list(GRADE["min_samples_leaf"])}
NOME_HP = {"n_estimators": "n_estimators", "profundidade": "max_depth", "min_samples_leaf": "min_samples_leaf"}
def ruido(col):
    if col.startswith("cv_") and f"{col}_ep" in df_cfg: return float(df_cfg[f"{col}_ep"].mean())
    if col.startswith("val_") and f"val_dp_{col[4:]}" in df_cfg: return float(df_cfg[f"val_dp_{col[4:]}"].mean() / np.sqrt(len(SEMENTES)))
    return float("nan")

linhas_ef, linhas_amp = [], []
for fator, niveis in NIVEIS.items():
    for nv in niveis:
        sub = df_cfg[df_cfg[fator] == nv]; linhas_ef.append({"hiperparametro": NOME_HP[fator], "valor": nv, **{r: sub[c].mean() for r, c in INFL}})
tab_ef = pd.DataFrame(linhas_ef)
for fator in NIVEIS:
    sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
    for r, c in INFL:
        amp = float(sub[r].max() - sub[r].min()); rz = ruido(c)
        leitura = "efeito no custo" if rz != rz else ("ACIMA do ruído" if amp > 2 * rz else "dentro do ruído")
        linhas_amp.append({"hiperparametro": NOME_HP[fator], "métrica": r, "amplitude": amp, "ruído (erro-padrão)": rz, "amplitude / ruído": amp / rz if rz == rz and rz > 0 else float("nan"), "leitura": leitura})
tab_amp = pd.DataFrame(linhas_amp)
print("== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre as combinações dos outros dois) ==")
display(tab_ef.round(5))
print("\n== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==")
display(tab_amp.pivot(index="métrica", columns="hiperparametro", values="amplitude").loc[[r for r, _ in INFL]][[NOME_HP[f] for f in NIVEIS]].round(5))
display(tab_amp[tab_amp["métrica"].isin(["ROC-AUC (CV)", "MCC proporção real (CV)"])].round(4))

# um de cada vez, a partir do padrão
base = df_cfg[(df_cfg.n_estimators == PADRAO["n_estimators"]) & (df_cfg.profundidade == rot(PADRAO["max_depth"])) & (df_cfg.min_samples_leaf == PADRAO["min_samples_leaf"])]
tab_um = pd.DataFrame()
if len(base):
    b0 = base.iloc[0]; linhas_um = []
    for _, r in df_cfg.iterrows():
        difs = [f for f in NIVEIS if r[f] != b0[f]]
        if len(difs) == 1:
            f = difs[0]; linhas_um.append({"muda": NOME_HP[f], "de": b0[f], "para": r[f], **{f"Δ {n}": r[c] - b0[c] for n, c in INFL}, "Δ ROC-AUC (CV) em erros-padrão": (r["cv_roc_auc"] - b0["cv_roc_auc"]) / b0["cv_roc_auc_ep"],
                              "Δ MCC real (CV) em erros-padrão": (r["cv_mcc_real"] - b0["cv_mcc_real"]) / b0["cv_mcc_real_ep"]})
    tab_um = pd.DataFrame(linhas_um); print("\n== Um de cada vez, a partir do padrão (100 árvores, sem limite de profundidade, folha 1) ==")
    display(tab_um[["muda", "de", "para", "Δ ROC-AUC (CV)", "Δ MCC proporção real (CV)", "Δ ROC-AUC (CV) em erros-padrão", "Δ MCC real (CV) em erros-padrão", "Δ nós totais", "Δ tempo de treino (s)"]].round(5))
else: print("\n(o padrão do notebook 05 não está na grade: tabela 'um de cada vez' omitida)")

# decomposição da variação (três fatores, uma medida por célula)
def decompor(Y):
    a, b, c = Y.shape; mu = Y.mean(); ma, mb, mc = Y.mean((1, 2)), Y.mean((0, 2)), Y.mean((0, 1)); mab, mac, mbc = Y.mean(2), Y.mean(1), Y.mean(0)
    ss = {"n_estimators": b * c * ((ma - mu) ** 2).sum(), "max_depth": a * c * ((mb - mu) ** 2).sum(), "min_samples_leaf": a * b * ((mc - mu) ** 2).sum(),
          "n_estimators × max_depth": c * ((mab - ma[:, None] - mb[None, :] + mu) ** 2).sum(), "n_estimators × min_samples_leaf": b * ((mac - ma[:, None] - mc[None, :] + mu) ** 2).sum(),
          "max_depth × min_samples_leaf": a * ((mbc - mb[:, None] - mc[None, :] + mu) ** 2).sum()}
    res = Y - mab[:, :, None] - mac[:, None, :] - mbc[None, :, :] + ma[:, None, None] + mb[None, :, None] + mc[None, None, :] - mu
    ss["interação tripla (resíduo)"] = (res ** 2).sum(); tot = ((Y - mu) ** 2).sum()
    return {k: (float(v / tot) if tot > 0 else float("nan")) for k, v in ss.items()}
def cubo(col):
    Y = np.zeros((len(NIVEIS["n_estimators"]), len(NIVEIS["profundidade"]), len(NIVEIS["min_samples_leaf"])))
    for i, n in enumerate(NIVEIS["n_estimators"]):
        for j, p in enumerate(NIVEIS["profundidade"]):
            for k, l in enumerate(NIVEIS["min_samples_leaf"]): Y[i, j, k] = float(df_cfg[(df_cfg.n_estimators == n) & (df_cfg.profundidade == p) & (df_cfg.min_samples_leaf == l)][col].iloc[0])
    return Y
decomp = pd.DataFrame({r: decompor(cubo(c)) for r, c in INFL}); decomp.index.name = "fonte da variação"
print("\n== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==")
display((decomp[["ROC-AUC (CV)", "MCC proporção real (CV)", "ROC-AUC (validação)", "nós totais", "tempo de treino (s)"]] * 100).round(1).add_suffix(" (%)"))

# leitura automática
print("\n== Leitura automática (ROC-AUC e MCC na proporção real da validação cruzada) ==")
for fator in NIVEIS:
    for r in ("ROC-AUC (CV)", "MCC proporção real (CV)"):
        t = tab_amp[(tab_amp.hiperparametro == NOME_HP[fator]) & (tab_amp["métrica"] == r)].iloc[0]; sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        melhor = sub.loc[sub[r].idxmax(), "valor"]
        print(f"  {NOME_HP[fator]:17s} {r:25s} amplitude {t['amplitude']:.5f} ({t['amplitude / ruído']:.1f}× o ruído) → {t['leitura']}; melhor valor na média: {melhor}")

# gráficos
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(2, 3, figsize=(12, 6.2))
    for j, fator in enumerate(NIVEIS):
        sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        for i, (r, c) in enumerate((("ROC-AUC (CV)", "cv_roc_auc"), ("MCC proporção real (CV)", "cv_mcc_real"))):
            ax[i, j].errorbar(range(len(sub)), sub[r], yerr=ruido(c), marker="o", capsize=4); ax[i, j].set_xticks(range(len(sub))); ax[i, j].set_xticklabels([str(v) for v in sub["valor"]]); ax[i, j].grid(alpha=.3)
            ax[i, j].set_title(f"{NOME_HP[fator]}" if i == 0 else ""); ax[i, j].set_ylabel(r if j == 0 else ""); ax[i, j].set_xlabel(NOME_HP[fator] if i == 1 else "")
    fig.suptitle("Efeito principal de cada hiperparâmetro (média sobre os outros; barra = 1 erro-padrão típico)"); fig.tight_layout(); fig.savefig(PASTA / "influencia_efeitos_principais.png", dpi=130); plt.close(fig)
    df_cfg["nos_por_arvore"] = df_cfg["nos"] / df_cfg["n_estimators"]; fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for a_, (r, c) in zip(ax, (("ROC-AUC (CV)", "cv_roc_auc"), ("MCC proporção real (CV)", "cv_mcc_real"), ("nós por árvore", "nos_por_arvore"))):
        pv = df_cfg.pivot_table(index="profundidade", columns="min_samples_leaf", values=c, aggfunc="mean").reindex(index=NIVEIS["profundidade"], columns=NIVEIS["min_samples_leaf"])
        im = a_.imshow(pv.values, aspect="auto", cmap="viridis"); a_.set_xticks(range(pv.shape[1])); a_.set_xticklabels(pv.columns); a_.set_yticks(range(pv.shape[0])); a_.set_yticklabels(pv.index)
        a_.set_xlabel("min_samples_leaf"); a_.set_ylabel("max_depth"); a_.set_title(r); plt.colorbar(im, ax=a_)
        for (ii, jj), v in np.ndenumerate(pv.values): a_.text(jj, ii, f"{v:.4g}", ha="center", va="center", color="w", fontsize=8)
    fig.suptitle("Interação max_depth × min_samples_leaf (média sobre n_estimators)"); fig.tight_layout(); fig.savefig(PASTA / "influencia_interacao.png", dpi=130); plt.close(fig)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a_, fonte, tit in zip(ax, ("cv", "validacao"), ("validação cruzada", "validação separada")):
        for (d, l) in PARES:
            t = df_curva[(df_curva.fonte == fonte) & (df_curva.profundidade == rot(d)) & (df_curva.min_samples_leaf == l)].groupby("n_arvores")["roc_auc"].mean(); a_.plot(t.index, t.values, marker="o", ms=3, label=f"prof={rot(d)} folha={l}")
        a_.set_xlabel("número de árvores"); a_.set_ylabel("ROC-AUC"); a_.set_title(f"ROC-AUC × árvores ({tit})"); a_.grid(alpha=.3)
    ax[1].legend(fontsize=7, ncol=2); fig.tight_layout(); fig.savefig(PASTA / "convergencia_arvores.png", dpi=130); plt.close(fig); print("\n✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png, convergencia_arvores.png")
except ImportError:
    print("\n(matplotlib não está instalado: gráficos não gerados)")

== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre as combinações dos outros dois) ==


,hiperparametro,valor,ROC-AUC (CV),MCC proporção real (CV),recall (CV),falso alarme (CV),ROC-AUC (validação),MCC proporção real (validação),recall com 1% de falso alarme (validação),nós totais,tamanho do modelo (MiB),tempo de treino (s),tempo de previsão na validação (s)
0,n_estimators,100,0.99997,0.85894,0.99771,0.00157,0.99998,0.88536,0.99984,25110.96296,0.56432,4.77769,0.28750
1,n_estimators,300,0.99997,0.86153,0.99775,0.00157,0.99999,0.88896,0.99986,75228.00000,1.68824,14.33307,0.68711
2,n_estimators,500,0.99997,0.86065,0.99773,0.00154,0.99999,0.88885,0.99986,125129.25926,2.80659,23.88846,1.07828
3,max_depth,None,0.99997,0.89576,0.99843,0.00120,0.99999,0.92282,0.99989,94616.14815,2.03719,15.13897,0.70380
4,max_depth,15,0.99998,0.89350,0.99839,0.00120,0.99999,0.92297,0.99990,89212.37037,1.95737,14.90707,0.66534
5,max_depth,8,0.99996,0.79185,0.99637,0.00227,0.99998,0.81737,0.99977,41639.70370,1.06458,12.95317,0.68376
6,min_samples_leaf,1,0.99997,0.88708,0.99820,0.00142,0.99999,0.90252,0.99993,95065.18519,1.91425,14.63453,0.69838
7,min_samples_leaf,5,0.99998,0.85482,0.99764,0.00152,0.99999,0.88860,0.99986,70238.51852,1.63563,14.27380,0.68025
8,min_samples_leaf,10,0.99997,0.83922,0.99734,0.00175,0.99998,0.87204,0.99977,60164.51852,1.50927,14.09089,0.67426



== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==


hiperparametro,n_estimators,max_depth,min_samples_leaf
métrica,,,
ROC-AUC (CV),0.00000,0.00002,0.00001
MCC proporção real (CV),0.00259,0.10391,0.04786
recall (CV),0.00005,0.00206,0.00086
falso alarme (CV),0.00003,0.00107,0.00033
ROC-AUC (validação),0.00000,0.00001,0.00000
MCC proporção real (validação),0.00360,0.10560,0.03048
recall com 1% de falso alarme (validação),0.00002,0.00013,0.00015
nós totais,100018.29630,52976.44444,34900.66667
tamanho do modelo (MiB),2.24227,0.97261,0.40498


,hiperparametro,métrica,amplitude,ruído (erro-padrão),amplitude / ruído,leitura
0,n_estimators,ROC-AUC (CV),0.0000,0.0000,0.3664,dentro do ruído
1,n_estimators,MCC proporção real (CV),0.0026,0.0166,0.1562,dentro do ruído
11,max_depth,ROC-AUC (CV),0.0000,0.0000,1.9793,dentro do ruído
12,max_depth,MCC proporção real (CV),0.1039,0.0166,6.2592,ACIMA do ruído
22,min_samples_leaf,ROC-AUC (CV),0.0000,0.0000,0.5525,dentro do ruído
23,min_samples_leaf,MCC proporção real (CV),0.0479,0.0166,2.8829,ACIMA do ruído



== Um de cada vez, a partir do padrão (100 árvores, sem limite de profundidade, folha 1) ==


,muda,de,para,Δ ROC-AUC (CV),Δ MCC proporção real (CV),Δ ROC-AUC (CV) em erros-padrão,Δ MCC real (CV) em erros-padrão,Δ nós totais,Δ tempo de treino (s)
0,max_depth,None,15,0.00003,-0.00626,0.71809,-0.27896,-4080.66667,-0.24570
1,max_depth,None,8,0.00001,-0.12856,0.21605,-5.73142,-27594.66667,-0.99733
2,min_samples_leaf,1,5,0.00002,-0.04333,0.59225,-1.93199,-13785.33333,-0.25387
3,min_samples_leaf,1,10,0.00002,-0.06463,0.56830,-2.88123,-18540.66667,-0.48473
4,n_estimators,100,300,0.00001,0.00397,0.31544,0.17694,84971.33333,10.58504
5,n_estimators,100,500,0.00001,0.00221,0.31216,0.09841,169610.00000,21.17009



== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==


,ROC-AUC (CV) (%),MCC proporção real (CV) (%),ROC-AUC (validação) (%),nós totais (%),tempo de treino (s) (%)
fonte da variação,,,,,
n_estimators,2.8,0.0,5.2,59.7,97.7
max_depth,65.7,83.2,45.7,20.3,1.5
min_samples_leaf,5.2,14.1,2.7,7.7,0.1
n_estimators × max_depth,0.1,0.0,9.1,6.0,0.5
n_estimators × min_samples_leaf,1.1,0.0,9.5,2.3,0.0
max_depth × min_samples_leaf,23.6,2.7,9.4,3.1,0.1
interação tripla (resíduo),1.5,0.0,18.4,0.9,0.0



== Leitura automática (ROC-AUC e MCC na proporção real da validação cruzada) ==
  n_estimators      ROC-AUC (CV)              amplitude 0.00000 (0.4× o ruído) → dentro do ruído; melhor valor na média: 300
  n_estimators      MCC proporção real (CV)   amplitude 0.00259 (0.2× o ruído) → dentro do ruído; melhor valor na média: 300
  max_depth         ROC-AUC (CV)              amplitude 0.00002 (2.0× o ruído) → dentro do ruído; melhor valor na média: 15
  max_depth         MCC proporção real (CV)   amplitude 0.10391 (6.3× o ruído) → ACIMA do ruído; melhor valor na média: None
  min_samples_leaf  ROC-AUC (CV)              amplitude 0.00001 (0.6× o ruído) → dentro do ruído; melhor valor na média: 5
  min_samples_leaf  MCC proporção real (CV)   amplitude 0.04786 (2.9× o ruído) → ACIMA do ruído; melhor valor na média: 1

✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png, convergencia_arvores.png


## 7. Recall por tipo de ataque
O recall de cada tipo de ataque (limiar 0,5), na validação cruzada e na validação separada, para a configuração **escolhida** e para o **padrão**. Na primeira linha, o BENIGN mostra a **especificidade** (BENIGN reconhecido como BENIGN). Um recall médio alto pode esconder um tipo mal detectado: é isto que esta tabela revela. As tabelas completas (todas as 27 configurações) vão para `recall_por_tipo_grade.csv`.

In [13]:
# ---------- 7.1 Recall por tipo ----------
tab_tipo = df_tipo.groupby(CHAVE + ["fonte", "tipo"])["recall"].mean().reset_index()
def _rt(n, p, l): return tab_tipo[(tab_tipo.n_estimators == n) & (tab_tipo.profundidade == p) & (tab_tipo.min_samples_leaf == l)].pivot(index="tipo", columns="fonte", values="recall").reindex(TIPOS)
rt_esc = _rt(N_ESC, ESC["profundidade"], L_ESC); out = pd.DataFrame({"n no ajuste": tab_tipos["ajuste"], "n na validação": tab_tipos["validação"]})
out["escolhida: CV"], out["escolhida: validação"] = rt_esc["cv_fora_da_dobra"], rt_esc["validacao"]
if len(pd_):
    rt_pad = _rt(PADRAO["n_estimators"], rot(PADRAO["max_depth"]), PADRAO["min_samples_leaf"]); out["padrão: CV"], out["padrão: validação"] = rt_pad["cv_fora_da_dobra"], rt_pad["validacao"]
print(f"== Recall por tipo de ataque (limiar {LIMIAR}; média das sementes) | BENIGN = especificidade ==")
display(out.round(5))
pior = rt_esc["validacao"].drop("BENIGN").idxmin(); print(f"Tipo com menor recall na validação (escolhida): {pior} ({rt_esc.loc[pior, 'validacao']:.4f}; {int(tab_tipos.loc[pior, 'validação'])} linhas na validação)")

== Recall por tipo de ataque (limiar 0.5; média das sementes) | BENIGN = especificidade ==


,n no ajuste,n na validação,escolhida: CV,escolhida: validação,padrão: CV,padrão: validação
BENIGN,40614,10154,0.99897,0.99915,0.99894,0.99921
DNS,3662,4538,0.99745,0.99581,0.99709,0.99559
LDAP,3662,4538,0.99782,0.99956,0.99863,0.99956
MSSQL,3662,4538,1.00000,0.99978,1.00000,0.99978
NTP,3662,4538,1.00000,0.99963,1.00000,0.99978
NetBIOS,3662,4537,0.99809,0.99794,0.99800,0.99802
SNMP,3662,4538,0.99736,0.99978,0.99727,0.99978
SSDP,3662,4538,1.00000,1.00000,1.00000,1.00000
Syn,3662,4538,1.00000,0.99978,1.00000,0.99978
TFTP,3663,4538,0.99973,0.99978,0.99973,0.99978


Tipo com menor recall na validação (escolhida): WebDDoS (0.9639; 83 linhas na validação)


## 8. A combinação escolhida: treino no ajuste e avaliação na validação
Treina a combinação escolhida no ajuste inteiro (semente 42, como no notebook 05) e repete as avaliações do notebook 05: métricas na validação (sem pesos e na proporção real), pontos de operação (**os limiares que o notebook 08 usará no teste**), dispersão entre 5 partições da validação, a validação cruzada (vinda da busca) e o **recall por tipo de ataque**.

In [14]:
# ---------- 8.1 Treino da combinação escolhida no ajuste ----------
print("Combinação escolhida:", HIPERPARAMETROS_FINAL)
with Medidor() as med_aj:
    rf = RandomForestClassifier(**HIPERPARAMETROS_FINAL).fit(X_aj, y_aj)
rf.colunas_modelo_ = list(CM)                                    # a ordem dos atributos viaja dentro do arquivo do modelo
fit_ajuste = {"relogio_s": med_aj.relogio, "cpu_s": med_aj.cpu, "pico_ram_mib": med_aj.pico_mib, "acrescimo_ram_mib": med_aj.delta_mib}
print(f"✔ RF treinado no ajuste: {med_aj.relogio:.1f} s de relógio, {med_aj.cpu:.1f} s de CPU, pico de RAM {med_aj.pico_mib:,.0f} MiB (+{med_aj.delta_mib:,.0f} MiB sobre o início)")

# ---------- 8.2 Métricas na validação ----------
p_va = rf.predict_proba(X_va)[:, 1]
w_prop = np.where(y_va == 0, PESO_BENIGN_PROP, 1.0)
m_sem, m_real = metricas(y_va, p_va, LIMIAR, None), metricas(y_va, p_va, LIMIAR, w_prop)
DEPENDE = {k: "sim" for k in ("acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign")}
ordem = ["roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign"]
tab_val = pd.DataFrame({"sem pesos": {k: m_sem[k] for k in ordem}, "proporção real": {k: m_real[k] for k in ordem}}); tab_val["depende da proporção?"] = [DEPENDE.get(k, "não") for k in tab_val.index]
print(f"\n== Métricas na validação (limiar {LIMIAR}) ==  (para as métricas que 'não' dependem da proporção, as duas colunas são iguais)")
display(tab_val.round(5))
print("Matriz de confusão (sem pesos): VP={tp:,.0f} FP={fp:,.0f} VN={tn:,.0f} FN={fn:,.0f}".format(**m_sem))
PONTOS = []
for alvo in FPR_ALVOS:
    op = recall_em_fpr(y_va, p_va, alvo); mo = metricas(y_va, p_va, op["limiar"], w_prop)
    PONTOS.append({"alvo_fpr": alvo, "limiar": op["limiar"], "recall": op["recall"], "fpr_obtida": op["fpr"], "precisao_real": mo["precisao"], "f1_real": mo["f1"], "mcc_real": mo["mcc"], "precisao_benign_real": mo["precisao_benign"], "f1_benign_real": mo["f1_benign"]})
tab_pontos = pd.DataFrame(PONTOS); print("\n== Ponto de operação: recall com falso alarme fixo (limiar escolhido na validação) ==")
display(tab_pontos.round(5))

# ---------- 8.3 Dispersão entre partições da validação ----------
linhas_part = []
for k, (_, idx) in enumerate(StratifiedKFold(K_PARTICOES_VAL, shuffle=True, random_state=SEMENTE).split(X_va, y_va), 1):
    m1, m2 = metricas(y_va[idx], p_va[idx], LIMIAR, None), metricas(y_va[idx], p_va[idx], LIMIAR, w_prop[idx])
    linhas_part.append({"particao": k, "n": len(idx), "benign": int((y_va[idx] == 0).sum()), "roc_auc": m1["roc_auc"], "recall": m1["recall"], "taxa_falso_alarme": m1["taxa_falso_alarme"],
                        "recall_fpr1pct": recall_em_fpr(y_va[idx], p_va[idx], 0.01)["recall"], "precisao_real": m2["precisao"], "f1_real": m2["f1"], "mcc_real": m2["mcc"], "pr_auc_real": m2["pr_auc"]})
tab_part = pd.DataFrame(linhas_part); resumo_part = tab_part.drop(columns=["particao", "n", "benign"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Dispersão entre {K_PARTICOES_VAL} partições da validação ==")
display(tab_part.round(5)); display(resumo_part.round(5))

# ---------- 8.4 Validação cruzada (da busca) para a combinação escolhida ----------
sel_cv = df_fold[(df_fold.n_estimators == N_ESC) & (df_fold.profundidade == ESC["profundidade"]) & (df_fold.min_samples_leaf == L_ESC)]
tab_cv = sel_cv.groupby("dobra")[["roc_auc", "recall", "taxa_falso_alarme", "acuracia_balanceada", "mcc", "mcc_real"]].mean().reset_index()
resumo_cv = tab_cv.drop(columns=["dobra"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Validação cruzada da combinação escolhida ({K_CV} dobras; média das {len(SEMENTES)} sementes em cada dobra) ==")
display(tab_cv.round(5)); display(resumo_cv.round(5))

# ---------- 8.5 Recall por tipo de ataque (validação, modelo do ajuste) ----------
tab_tipo_val = pd.DataFrame({"n na validação": np.bincount(g_va, minlength=len(TIPOS)), f"limiar {LIMIAR}": recall_por_tipo(y_va, p_va, g_va, LIMIAR, len(TIPOS))}, index=TIPOS)
for pt in PONTOS: tab_tipo_val[f"limiar p/ falso alarme {pt['alvo_fpr']:.1%} ({pt['limiar']:.3f})"] = recall_por_tipo(y_va, p_va, g_va, pt["limiar"], len(TIPOS))
print("\n== Recall por tipo de ataque na validação (BENIGN = especificidade) ==")
display(tab_tipo_val.round(5))

Combinação escolhida: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}
✔ RF treinado no ajuste: 16.2 s de relógio, 112.9 s de CPU, pico de RAM 365 MiB (+75 MiB sobre o início)

== Métricas na validação (limiar 0.5) ==  (para as métricas que 'não' dependem da proporção, as duas colunas são iguais)


,sem pesos,proporção real,depende da proporção?
roc_auc,0.99999,0.99999,não
recall,0.99918,0.99918,não
taxa_falso_alarme,0.00079,0.00079,não
especificidade,0.99921,0.99921,não
acuracia_balanceada,0.99920,0.99920,não
acuracia,0.99919,0.99918,sim
precisao,0.99984,1.00000,sim
f1,0.99951,0.99959,sim
mcc,0.99710,0.93886,sim
pr_auc,1.00000,1.00000,sim


Matriz de confusão (sem pesos): VP=49,959 FP=8 VN=10,146 FN=41

== Ponto de operação: recall com falso alarme fixo (limiar escolhido na validação) ==


,alvo_fpr,limiar,recall,fpr_obtida,precisao_real,f1_real,mcc_real,precisao_benign_real,f1_benign_real
0,0.010,0.04042,1.00000,0.00906,0.99994,0.99997,0.99543,1.00000,0.99545
1,0.001,0.41599,0.99952,0.00098,0.99999,0.99976,0.96258,0.92793,0.96216



== Dispersão entre 5 partições da validação ==


,particao,n,benign,roc_auc,recall,taxa_falso_alarme,recall_fpr1pct,precisao_real,f1_real,mcc_real,pr_auc_real
0,1,12031,2031,1.00000,0.9993,0.00000,1.0,1.00000,0.99965,0.94749,1.0
1,2,12031,2031,0.99999,0.9994,0.00098,1.0,0.99999,0.99970,0.95397,1.0
2,3,12031,2031,1.00000,0.9988,0.00000,1.0,1.00000,0.99940,0.91463,1.0
3,4,12031,2031,1.00000,0.9995,0.00098,1.0,0.99999,0.99975,0.96113,1.0
4,5,12030,2030,0.99999,0.9989,0.00197,1.0,0.99999,0.99944,0.91984,1.0


... (2 outputs left out)

,dobra,roc_auc,recall,taxa_falso_alarme,acuracia_balanceada,mcc,mcc_real
0,1,0.99999,0.99778,0.00127,0.99826,0.99651,0.85646
1,2,1.00000,0.99947,0.00049,0.99949,0.99897,0.95900
2,3,0.99997,0.99947,0.00135,0.99906,0.99811,0.95855
3,4,1.00000,0.99955,0.00107,0.99924,0.99848,0.96466
4,5,1.00000,0.99860,0.00094,0.99883,0.99766,0.90225


,mean,std,min,max
roc_auc,0.99999,0.00001,0.99997,1.00000
recall,0.99897,0.00077,0.99778,0.99955
taxa_falso_alarme,0.00103,0.00034,0.00049,0.00135
acuracia_balanceada,0.99897,0.00047,0.99826,0.99949
mcc,0.99795,0.00094,0.99651,0.99897
mcc_real,0.92818,0.04748,0.85646,0.96466



== Recall por tipo de ataque na validação (BENIGN = especificidade) ==


,n na validação,limiar 0.5,limiar p/ falso alarme 1.0% (0.040),limiar p/ falso alarme 0.1% (0.416)
BENIGN,10154,0.99921,0.99094,0.99902
DNS,4538,0.99559,1.00000,0.99780
LDAP,4538,0.99956,1.00000,0.99956
MSSQL,4538,0.99978,1.00000,0.99978
NTP,4538,0.99956,1.00000,1.00000
NetBIOS,4537,0.99802,1.00000,0.99890
SNMP,4538,0.99978,1.00000,0.99978
SSDP,4538,1.00000,1.00000,1.00000
Syn,4538,0.99978,1.00000,0.99978
TFTP,4538,0.99978,1.00000,0.99978


## 9. Custo computacional do modelo do ajuste
Tamanho em disco, número de nós e profundidade, e a latência (1 fluxo por vez) e a vazão (lote) com `n_jobs = 1` e com todos os núcleos, com e sem o pré-processamento. Mesma medição do notebook 05.

In [15]:
# ---------- 9.1 Custo do modelo da fase 1 ----------
tam_aj = tamanho_e_nos(rf, PASTA / "modelo_ajuste.joblib", COMPRESSAO_MODELO)
inf_aj = medir_inferencia(rf, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"Modelo do ajuste: {tam_aj['arquivo_mib']:.1f} MiB em disco | {tam_aj['n_arvores']} árvores | {tam_aj['nos_total']:,} nós | profundidade média {tam_aj['profundidade_media']:.1f} (máx {tam_aj['profundidade_max']})")
display(pd.DataFrame(inf_aj).set_index("n_jobs").T.round(3))

Modelo do ajuste: 2.2 MiB em disco | 300 árvores | 114,544 nós | profundidade média 15.0 (máx 15)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,31.348,116.426
latencia_modelo_p95_ms,89.877,236.020
latencia_modelo_p99_ms,128.527,338.295
latencia_fim_a_fim_p50_ms,31.554,116.590
latencia_fim_a_fim_p95_ms,90.231,236.134
latencia_fim_a_fim_p99_ms,128.868,338.537
vazao_modelo_fluxos_s,36947.968,144104.321
vazao_fim_a_fim_fluxos_s,44826.588,121039.417
n_lote,60154.000,60154.000


## 10. Refit (modelo final)
Treina de novo a combinação escolhida com o **mesmo conjunto do refit do notebook 05** (todos os BENIGN do treino e o mesmo número de ataques), repetido `REPETICOES_TEMPO` vezes para medir tempo, CPU e RAM. Esse é o modelo que o notebook 08 avalia no teste.

In [16]:
# ---------- 10.1 Refit ----------
rf_df = ordenar(ler_view(V_REFIT)); X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int); assert np.isfinite(X_rf).all(), "valores não finitos no refit"
print(f"Refit: {len(X_rf):,} linhas ({int((y_rf == 0).sum()):,} BENIGN, {int((y_rf == 1).sum()):,} ataques)")
reps, modelo_final, p_ref = [], None, None
for k in range(1, REPETICOES_TEMPO + 1):
    with Medidor() as m:
        mdl = RandomForestClassifier(**HIPERPARAMETROS_FINAL).fit(X_rf, y_rf)
    reps.append({"repeticao": k, "relogio_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib})
    if modelo_final is None:
        modelo_final = mdl; modelo_final.colunas_modelo_ = list(CM); p_ref = mdl.predict_proba(X_va[:5000])[:, 1]
    else:
        assert np.allclose(mdl.predict_proba(X_va[:5000])[:, 1], p_ref), "as repetições do refit deram modelos diferentes (a semente não está fixando o resultado)"
    del mdl
tab_reps = pd.DataFrame(reps); display(tab_reps.round(3))
fit_refit = {"repeticoes": REPETICOES_TEMPO, "relogio_mediana_s": float(tab_reps.relogio_s.median()), "relogio_min_s": float(tab_reps.relogio_s.min()), "relogio_max_s": float(tab_reps.relogio_s.max()),
             "cpu_mediana_s": float(tab_reps.cpu_s.median()), "pico_ram_max_mib": float(tab_reps.pico_ram_mib.max()), "acrescimo_ram_max_mib": float(tab_reps.acrescimo_ram_mib.max())}
tam_fin = tamanho_e_nos(modelo_final, PASTA / "modelo_final_refit.joblib", COMPRESSAO_MODELO)
inf_fin = medir_inferencia(modelo_final, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"\n✔ Refit: {fit_refit['relogio_mediana_s']:.1f} s (mediana de {REPETICOES_TEMPO}; mín {fit_refit['relogio_min_s']:.1f}, máx {fit_refit['relogio_max_s']:.1f}), CPU {fit_refit['cpu_mediana_s']:.1f} s, pico de RAM {fit_refit['pico_ram_max_mib']:,.0f} MiB")
print(f"Modelo final: {tam_fin['arquivo_mib']:.1f} MiB em disco | {tam_fin['nos_total']:,} nós | profundidade média {tam_fin['profundidade_media']:.1f} (máx {tam_fin['profundidade_max']})")
print("(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)")
display(pd.DataFrame(inf_fin).set_index("n_jobs").T.round(3))

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
Refit: 101,536 linhas (50,768 BENIGN, 50,768 ataques)


,repeticao,relogio_s,cpu_s,pico_ram_mib,acrescimo_ram_mib
0,1,20.912,152.812,606.438,47.922
1,2,20.905,152.594,615.727,94.590
2,3,20.401,149.578,615.863,36.801



✔ Refit: 20.9 s (mediana de 3; mín 20.4, máx 20.9), CPU 152.6 s, pico de RAM 616 MiB
Modelo final: 2.5 MiB em disco | 125,006 nós | profundidade média 15.0 (máx 15)
(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,26.579,102.193
latencia_modelo_p95_ms,32.490,124.722
latencia_modelo_p99_ms,39.518,145.751
latencia_fim_a_fim_p50_ms,26.672,102.333
latencia_fim_a_fim_p95_ms,32.599,124.882
latencia_fim_a_fim_p99_ms,39.645,145.854
vazao_modelo_fluxos_s,31484.510,143617.630
vazao_fim_a_fim_fluxos_s,42128.314,121370.379
n_lote,60154.000,60154.000


## 11. Gravação dos registros
Grava na pasta da execução tudo o que o notebook 08 e o TCC vão precisar: o modelo final, o pré-processamento, as métricas e os limiares da validação, o custo, e as tabelas da busca (todas as configurações, o efeito de cada hiperparâmetro, o recall por tipo). Acrescenta uma linha ao `registro_buscas.csv`.

In [17]:
# ---------- 11.1 Gravação ----------
hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
protocolo = {"k_particoes_validacao": K_PARTICOES_VAL, "k_cv_ajuste": K_CV, "repeticoes_tempo_refit": REPETICOES_TEMPO, "n_latencia": N_LATENCIA, "fpr_alvos": list(FPR_ALVOS)}
config = {"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "modelo": "RandomForestClassifier", "variante": "sem_auc_ajustado", "origem": "notebook 07 (busca de hiperparâmetros)",
          "hiperparametros": HIPERPARAMETROS_FINAL, "semente": SEMENTE, "limiar_padrao": LIMIAR, "versao_selecao_notebook04": versao_sel, "versao_views_notebook03": versao_views,
          "dados": {"ajuste": len(X_aj), "validacao": len(X_va), "refit": len(X_rf), "benign_treino": N_BENIGN, "ataques_treino": N_ATAQUE, "peso_benign_proporcao_real": PESO_BENIGN_PROP},
          "protocolo": protocolo, "software": sw, "hardware": hw}
salvar_json(config, PASTA / "config_treino.json"); salvar_json(prep.para_dict(), PASTA / "preprocessamento.json")
_criado_em = json.loads((PASTA / "config_busca.json").read_text(encoding="utf-8")).get("criado_em")
cfg_busca = {"run_id": RUN_ID, "impressao": IMPRESSAO, "retomada": RETOMANDO, "criado_em": _criado_em, "concluido_em": datetime.now().isoformat(timespec="seconds"),
             "tempo_busca": {"relogio_s": TEMPO_BUSCA_RELOGIO, "sessoes": len(tempos_sessoes), "treino_somado_s": TEMPO_TREINO_SOMADO, "florestas": N_FLORESTAS, "detalhe_sessoes": tempos_sessoes.to_dict(orient="records")}, "grade": GRADE, "sementes": SEMENTES, "k_cv": K_CV, "semente_dobras": SEMENTE,
             "tamanhos_avaliados": TAM_TODOS, "n_florestas": N_FLORESTAS, "fixos": FIXOS, "regra": {"metricas": METRICAS_DECISAO, "mult_erro_padrao": MULT_ERRO_PADRAO, "desempate": "menos nós, depois menos árvores, depois folha maior"},
             "tipos": TIPOS, "software": sw, "hardware": hw}
salvar_json(cfg_busca, PASTA / "config_busca.json")
salvar_json({"escolhida": {"n_estimators": N_ESC, "max_depth": D_ESC, "min_samples_leaf": L_ESC, "config": ESC["config"]}, "usou_plano_b": bool(FALLBACK), "passo1": INFO_SEL, "n_elegiveis": len(CAND),
             "elegiveis": CAND[["config", "cv_roc_auc", "cv_mcc_real", "nos"]].to_dict(orient="records"), "hiperparametros_finais": HIPERPARAMETROS_FINAL}, PASTA / "selecao_hiperparametros.json")
df_cfg.to_csv(PASTA / "resultados_grade.csv", index=False); df_fold.to_csv(PASTA / "cv_dobras.csv", index=False); df_oof.to_csv(PASTA / "cv_fora_da_dobra_agregado.csv", index=False)
df_val.to_csv(PASTA / "validacao_por_config.csv", index=False); tab_tipo.to_csv(PASTA / "recall_por_tipo_grade.csv", index=False); tab_comp.to_csv(PASTA / "comparacao_escolhida_padrao.csv", index=False)
tab_ef.to_csv(PASTA / "efeitos_principais.csv", index=False); tab_amp.to_csv(PASTA / "amplitude_efeitos.csv", index=False); decomp.reset_index().to_csv(PASTA / "decomposicao_variacao.csv", index=False)
df_curva.to_csv(PASTA / "convergencia_arvores.csv", index=False); tab_tipo_val.reset_index(names="tipo").to_csv(PASTA / "recall_por_tipo_validacao_final.csv", index=False)
if len(tab_um): tab_um.to_csv(PASTA / "efeito_um_de_cada_vez.csv", index=False)
met_val = {"limiar": LIMIAR, "sem_pesos": m_sem, "proporcao_real": m_real, "pontos_de_operacao": PONTOS, "dispersao_particoes": {"k": K_PARTICOES_VAL, "resumo": resumo_part.to_dict(orient="index")},
           "cv_ajuste": {"k": K_CV, "origem": "busca (média das sementes por dobra)", "resumo": resumo_cv.to_dict(orient="index")}}
salvar_json(met_val, PASTA / "metricas_validacao.json"); tab_val.assign(limiar=LIMIAR).reset_index(names="metrica").to_csv(PASTA / "metricas_validacao.csv", index=False)
tab_part.to_csv(PASTA / "dispersao_particoes_validacao.csv", index=False); tab_cv.to_csv(PASTA / "cv_ajuste.csv", index=False)
salvar_json({"limiar_padrao": LIMIAR, "pontos_de_operacao_validacao": PONTOS}, PASTA / "limiares_validacao.json")
custo = {"fit_ajuste": fit_ajuste, "fit_refit": fit_refit, "fit_refit_repeticoes": reps, "modelo_ajuste": tam_aj, "modelo_final": tam_fin, "inferencia_modelo_ajuste": inf_aj, "inferencia_modelo_final": inf_fin}
salvar_json(custo, PASTA / "custo_computacional.json")
linhas_custo = [{"item": "fit_ajuste", **fit_ajuste}, {"item": "fit_refit", **fit_refit}] + [{"item": f"inferencia_final_njobs{r['n_jobs']}", **r} for r in inf_fin] + [{"item": "modelo_final", **tam_fin}]
pd.DataFrame(linhas_custo).to_csv(PASTA / "custo_computacional.csv", index=False)
i1 = next(r for r in inf_fin if r["n_jobs"] == 1); iN = inf_fin[-1]
registro = {"run_id": RUN_ID, "data": config["data"], "retomada": RETOMANDO, "n_configuracoes": len(CONFIGS), "sementes": len(SEMENTES), "k_cv": K_CV, "escolhida_n_estimators": N_ESC, "escolhida_max_depth": rot(D_ESC),
            "escolhida_min_samples_leaf": L_ESC, "tempo_busca_relogio_s": TEMPO_BUSCA_RELOGIO, "tempo_treino_somado_s": TEMPO_TREINO_SOMADO, "n_sessoes": len(tempos_sessoes), "usou_plano_b": bool(FALLBACK), "n_elegiveis": len(CAND), "cv_roc_auc": ESC["cv_roc_auc"], "cv_roc_auc_ep": ESC["cv_roc_auc_ep"], "cv_mcc_real": ESC["cv_mcc_real"], "cv_mcc_real_ep": ESC["cv_mcc_real_ep"],
            "roc_auc_val": m_sem["roc_auc"], "recall_val": m_sem["recall"], "fpr_val": m_sem["taxa_falso_alarme"], "mcc_real_val": m_real["mcc"], "f1_benign_real_val": m_real["f1_benign"], "pr_auc_benign_real_val": m_real["pr_auc_benign"],
            "recall_fpr1pct_val": PONTOS[0]["recall"] if PONTOS else float("nan"), "fit_refit_relogio_mediana_s": fit_refit["relogio_mediana_s"], "modelo_final_mib": tam_fin["arquivo_mib"], "nos_total": tam_fin["nos_total"],
            "latencia_p50_ms_njobs1": i1["latencia_fim_a_fim_p50_ms"], "latencia_p99_ms_njobs1": i1["latencia_fim_a_fim_p99_ms"], "vazao_fluxos_s_njobs1": i1["vazao_fim_a_fim_fluxos_s"], "vazao_fluxos_s_njobs_orig": iN["vazao_fim_a_fim_fluxos_s"], "pasta": str(PASTA)}
caminho_reg = ARTEFATOS_BASE / "registro_buscas.csv"; pd.DataFrame([registro]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
print(f"✔ Artefatos gravados em {PASTA}\n✔ Registro acrescentado a {caminho_reg} ({sum(1 for _ in open(caminho_reg, encoding='utf-8')) - 1} busca(s) registrada(s))")
for f in sorted(PASTA.iterdir()):
    if f.is_file(): print(f"   {f.name:42s} {f.stat().st_size / 2**10:>10,.1f} KiB")

✔ Artefatos gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_busca\rf_busca_20261024_175105
✔ Registro acrescentado a C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_busca\registro_buscas.csv (1 busca(s) registrada(s))
   amplitude_efeitos.csv                             3.4 KiB
   comparacao_escolhida_padrao.csv                   1.3 KiB
   config_busca.json                                 1.7 KiB
   config_treino.json                                1.2 KiB
   convergencia_arvores.csv                         14.4 KiB
   convergencia_arvores.png                        102.7 KiB
   custo_computacional.csv                           1.1 KiB
   custo_computacional.json                          3.2 KiB
   cv_ajuste.csv                                     0.7 KiB
   cv_dobras.csv                                   104.8 KiB
   cv_fora_da_dobra_agregado.csv                    24.0 KiB
   decomposicao_variacao.csv               

## 12. Conferências finais
Confere que a busca e o modelo final estão corretos e reproduzíveis: unidades completas, dobras que cobrem o ajuste uma vez, o atalho das árvores igual ao treino direto, mesmas linhas do notebook 05, modelo gravado que reproduz as previsões, escolha reproduzível a partir do CSV, decomposição que soma 100%, e os arquivos esperados.

In [18]:
# ---------- 12.1 Conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

refit_sel = consultar("SELECT toString(id_linha) AS id_linha FROM " + TAB_REFIT)
verifica("Busca completa: todas as unidades gravadas", all((PASTA / "unidades" / f"{nome_unidade(*u)}.npz").exists() for u in UNIDADES), f"({len(UNIDADES)} unidades, {N_FLORESTAS} florestas)")
verifica("Dobras cobrem cada linha do ajuste exatamente uma vez", np.array_equal(np.sort(np.concatenate([ite for _, ite in DOBRAS])), np.arange(len(y_aj))))
verifica("Todo tipo de ataque aparece em todas as dobras", all(len(np.unique(g_aj[ite])) == len(TIPOS) for _, ite in DOBRAS), f"({len(TIPOS) - 1} tipos)")
_d0, _l0, _s0 = next(((d, l, s) for (d, l, s) in UNIDADES if s == SEMENTE), UNIDADES[0]); _k0 = TAM_GRADE[0]
_z = np.load(PASTA / "unidades" / f"{nome_unidade(_d0, _l0, _s0)}.npz"); _dir = RandomForestClassifier(n_estimators=_k0, max_depth=_d0, min_samples_leaf=_l0, random_state=_s0, **FIXOS).fit(X_aj, y_aj)
verifica(f"Atalho das árvores: as primeiras {_k0} de {NMAX} = floresta treinada com {_k0} (prof={rot(_d0)}, folha={_l0}, semente={_s0})", np.allclose(_z["val"][0], _dir.predict_proba(X_va)[:, 1], atol=1e-6), "(diferença máxima nas probabilidades: {:.1e})".format(float(np.abs(_z["val"][0] - _dir.predict_proba(X_va)[:, 1]).max())))
if SEMENTE in SEMENTES:
    _zz = np.load(PASTA / "unidades" / f"{nome_unidade(D_ESC, L_ESC, SEMENTE)}.npz")
    verifica("A combinação escolhida treinada direto reproduz a previsão da busca", np.allclose(_zz["val"][TAM_GRADE.index(N_ESC)], p_va, atol=1e-6))
m_ajuste_lido = joblib.load(PASTA / "modelo_ajuste.joblib"); m_final_lido = joblib.load(PASTA / "modelo_final_refit.joblib")
verifica("Modelo do ajuste gravado reproduz as previsões da validação", np.allclose(m_ajuste_lido.predict_proba(X_va)[:, 1], p_va))
verifica("Modelo final gravado reproduz as previsões do modelo em memória", np.allclose(m_final_lido.predict_proba(X_va[:5000])[:, 1], p_ref))
prep_lido = json.loads((PASTA / "preprocessamento.json").read_text(encoding="utf-8"))
verifica("Ordem dos atributos gravada = ordem usada no treino e dentro do modelo", prep_lido["colunas_modelo"] == CM == list(m_final_lido.colunas_modelo_) == list(m_ajuste_lido.colunas_modelo_))
verifica("Medianas gravadas = medianas da tabela de auditoria do notebook 03", prep_lido["medianas_treino"] == MEDIANAS and set(prep_lido["colunas_retiradas_ablacao"]) == set(RETIRADAS))
verifica("Hiperparâmetros do modelo final = combinação escolhida", all(getattr(m_final_lido, k) == v for k, v in HIPERPARAMETROS_FINAL.items()))
verifica("Ajuste balanceado 1:1", int((y_aj == 0).sum()) == int((y_aj == 1).sum()) == n_aj_b, f"({int((y_aj == 0).sum()):,} + {int((y_aj == 1).sum()):,})")
verifica("Refit com todos os BENIGN e balanceado", int((y_rf == 0).sum()) == N_BENIGN and int((y_rf == 1).sum()) == N_BENIGN, f"({int((y_rf == 0).sum()):,} + {int((y_rf == 1).sum()):,})")
verifica("Mesmo refit do notebook 05 (mesmas linhas)", set(rf_df["id_linha"]) == set(refit_sel["id_linha"]) and len(rf_df) == len(refit_sel), f"({len(rf_df):,} linhas)")
verifica("Refit contém todo o ajuste", set(aj["id_linha"]) <= set(rf_df["id_linha"]))
_esc2 = escolher(pd.read_csv(PASTA / "resultados_grade.csv"))[0]
verifica("A escolha é reproduzível a partir do resultados_grade.csv gravado", _esc2["config"] == ESC["config"], f"({ESC['config']})")
verifica("Decomposição da variação: cada coluna soma 100%", bool(np.allclose(decomp.sum(), 1.0, atol=1e-9)))
verifica("Probabilidades entre 0 e 1 e matriz de confusão fecha", bool(((p_va >= 0) & (p_va <= 1)).all()) and abs(m_sem["tp"] + m_sem["fp"] + m_sem["tn"] + m_sem["fn"] - len(y_va)) < 1e-9)
verifica("Validação ponderada tem a proporção BENIGN:ataque do treino", abs(float(w_prop[y_va == 0].sum()) / float(w_prop[y_va == 1].sum()) - N_BENIGN / N_ATAQUE) < 1e-9)
verifica("Métricas que não dependem da proporção são iguais nas duas colunas", all(abs(m_sem[k] - m_real[k]) < 1e-9 for k in ("roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada")))
verifica("Recall por tipo: todo tipo de ataque tem linhas na validação", bool((tab_tipos["validação"] > 0).all()))
verifica("Arquivos esperados gravados", all((PASTA / f).exists() for f in ("modelo_ajuste.joblib", "modelo_final_refit.joblib", "config_treino.json", "config_busca.json", "preprocessamento.json", "metricas_validacao.json", "metricas_validacao.csv",
                                                                       "dispersao_particoes_validacao.csv", "cv_ajuste.csv", "custo_computacional.json", "custo_computacional.csv", "limiares_validacao.json", "selecao_hiperparametros.json", "tempos_busca.csv",
                                                                       "resultados_grade.csv", "cv_dobras.csv", "validacao_por_config.csv", "recall_por_tipo_grade.csv", "efeitos_principais.csv", "decomposicao_variacao.csv", "convergencia_arvores.csv")))
verifica("Custo medido: tempos positivos" + ("" if psutil else " (memória não medida: psutil ausente)"), fit_ajuste["relogio_s"] > 0 and fit_refit["relogio_mediana_s"] > 0 and inf_fin[0]["vazao_fim_a_fim_fluxos_s"] > 0)
assert all(ok), "há checagens reprovadas acima"
(PASTA / "FINALIZADO").write_text(datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")
print(f"\nPróximo passo: notebook 08 (avaliação no teste, uma vez), apontando para {PASTA.name}")

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
✔ Busca completa: todas as unidades gravadas (27 unidades, 162 florestas)
✔ Dobras cobrem cada linha do ajuste exatamente uma vez 
✔ Todo tipo de ataque aparece em todas as dobras (12 tipos)
✔ Atalho das árvores: as primeiras 100 de 500 = floresta treinada com 100 (prof=None, folha=1, semente=42) (diferença máxima nas probabilidades: 2.9e-08)
✔ A combinação escolhida treinada direto reproduz a previsão da busca 
✔ Modelo do ajuste gravado reproduz as previsões da validação 
✔ Modelo final gravado reproduz as previsões do modelo em memória 
✔ Ordem dos atributos gravada = ordem usada no treino e dentro do modelo 
✔ Medianas gravadas = medianas da tabela de auditoria do notebook 03 
✔ Hiperparâmetros do modelo final = combinação escolhida 
✔ Ajuste balanceado 1:1 (40,614 + 40,614)
✔ Refit com todos os BENIGN e balanceado (50,768 + 50,768)
✔ Mesmo refit do notebook 05 (mesmas li